# Sector clustering for the stock market - coding <span style="color:#f56502; font-weight:bold">[20 points]</span>

**Reading Material**: Before addressing this exercise, we recommend that you study 
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=hierarchical-clustering">Clustering for the stock market</a> 

**Code Reference**: Become familiar with scripts
<a style="color:#3c9591" href="../Python/1. Textbook support code/Machine Learning/31d. Clustering for the stock market/s_shrink_corr_clusters.ipynb">s_shrink_corr_clusters</a>
that performs sector clustering for a set of stock values.

Consider the $286\times 1$ random vector of 
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=aosfiaod-copy-3" class="@formulas #aosfiaod-copy-3">daily log-returns</a>
$\boldsymbol{C}_{t}$ of the S&P 500 stocks stemming from the first $\bar{k}=6$ sectors.

<h1 style="font-weight: bold; font-size:14px; line-height:1.5em;">Prepare environment
</h1>

In [8]:
from numpy import (argsort, array, asarray, average, bincount, corrcoef, cumsum, diag, diagflat, diff,
                   eye, full, fill_diagonal, ix_, log, max, min, nan, ones, r_, real, sqrt, trace,
                   where, zeros, clip)
from numpy.linalg import det, eigvals
from numpy.random import multivariate_normal
from networkx import draw_networkx, from_numpy_array, kamada_kawai_layout, minimum_spanning_tree
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from pandas import read_csv
from seaborn import heatmap, lineplot, scatterplot
from matplotlib.pyplot import xticks, yticks

In [9]:
# --- PyCharm plumbing (not ARPM code) ---
import sys
from pathlib import Path
UNIT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "clustering_lab").is_dir())
if str(UNIT_ROOT) not in sys.path:
    sys.path.insert(0, str(UNIT_ROOT))
from clustering_lab.paths import stocks_csv_path
STOCKS_CSV = stocks_csv_path()
print("data:", STOCKS_CSV)

data: C:\Dev\Projects\ARPM\probabilistic_machine_learning\unit3_autoencoders_graphical_clustering\data\db_stocks_sp.csv


<h1 style="font-weight: bold; font-size:14px; line-height:1.5em;">Load data</h1>

From database
<a style="color:#3c9591" href="../../Databases/global-databases/equities/db_stocks_SP500/db_stocks_sp.csv">db_stocks_sp</a>
we upload the stock values along with the sector names and sectors indices.

In [10]:
db_stocks_sp = read_csv(STOCKS_CSV, index_col=0, header=[0, 1])
# all stock values
v_full = array(db_stocks_sp)
# all sector names
sectors_full = array(db_stocks_sp.columns.levels[0])
# sector index per column
labels_full = array(db_stocks_sp.columns.codes)[0, :]
# select only first 6 sectors
mask = labels_full < 6
v = v_full[:, mask]
# number of stocks after filtering
n_bar = v.shape[1]
# selected sector names (first 6 only)
sectors = sectors_full[:6]
# remap labels to match filtered sector set
labels = labels_full[mask]
# number of sectors after filtering
k_bar = sectors.shape[0]

print("stocks:", n_bar, " (expect 286)   sectors:", list(sectors))

stocks: 286  (expect 286)   sectors: ['ConsumerDiscretionary', 'ConsumerStaples', 'Energy', 'Financials', 'HealthCare', 'Industrials']


<h1 style="font-weight: bold; font-size:14px; line-height:1.5em;"><span style="color:#f56502; font-weight:bold">[9 points]</span> Task 1</h1>

Compute the 
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=SampCorrDeffEstim" class="@formulas #SampCorrDeffEstim">sample correlation</a>
$\hat{\boldsymbol{c}}^{2\operatorname{hist}}$ of the 
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=aosfiaod-copy-3" class="@formulas #aosfiaod-copy-3">log-returns</a>
$\boldsymbol{C}_{t}$ and the respective
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=Eucl_dist" class="@formulas #Eucl_dist">distance matrix</a>
$\boldsymbol{d}=\{d_{i,j}\}_{i,j}$.

In [11]:
# the daily log-returns (lnVt - lnVt -1)
# axis = 0 -> means get the differences 'down the rows' day by day for each stock
# find the differences between 2 values
eps = diff(log(v), axis = 0)

# getting the historical correlation of every pair of stocks
# .T(transpose) is needed because corrcoef treats rows as variables and the stocks are in columns
# returns an n_bar x n_bar table showing how strongly stocks move together, diagonals are all 1's
c2_hist = corrcoef(eps.T)

# distance matrix
# get the correlation-distance formula and apply it to the whole matrix at once
#
d = sqrt(clip(2*(1 - c2_hist), 0.0, None))
# forcing each stocks distance to itself to exactly 0
fill_diagonal(d, 0.0)

print("eps:", eps.shape, " c2_hist:", c2_hist.shape, " d:", d.shape)
print()
print("d diagonal max:", d.diagonal().max(), "  any nan in d:", bool((d != d).any()))
print()
print("d range:", d[~eye(n_bar, dtype=bool)].min().round(3), "→", d.max().round(3))

eps: (1599, 286)  c2_hist: (286, 286)  d: (286, 286)

d diagonal max: 0.0   any nan in d: False

d range: 0.392 → 1.331


<h1 style="font-weight: bold; font-size:14px; line-height:1.5em;"><span style="color:#f56502; font-weight:bold">[7 points]</span> Task 2</h1>

Sort the 
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=SampCorrDeffEstim" class="@formulas #SampCorrDeffEstim">sample correlation matrix</a>
$\hat{\boldsymbol{c}}^{2\operatorname{hist}}$ by sectors, and plot the corresponding heat map.

**Hint**: Use function
<a style="color:#3c9591" href="https://seaborn.pydata.org/generated/seaborn.heatmap.html">heatmap</a>
as in script
<a style="color:#3c9591" href="../Python/1. Textbook support code/Machine Learning/31d. Clustering for the stock market/s_shrink_corr_clusters.ipynb">s_shrink_corr_clusters</a>.

In [ ]:
# sorting index for sectors
i_s =  
# correlation matrix sorted by sectors
c2_sec =  

############################################### plot ###############################################
# sector clusters 
l_s = cumsum(bincount(labels[i_s]))
heatmap(c2_sec - eye(n_bar), vmin=min(c2_hist - eye(n_bar)), 
        vmax=max(c2_hist - eye(n_bar)), cmap = 'viridis', square=True)
[lineplot(x=[1, n_bar], y=[l_s[k], l_s[k]], c='r') for k in reversed(range(k_bar))]
[lineplot(x=[l_s[k], l_s[k]], y=[1, n_bar], estimator=None, c='r') for k in reversed(range(k_bar))]
[lineplot(x=[1, l_s[k]], y=[1, l_s[k]]) for k in reversed(range(k_bar))]
xticks(r_[l_s[0]/2, l_s[:-1] + diff(l_s)/2], sectors, rotation=90)
yticks(r_[l_s[0]/2, l_s[:-1] + diff(l_s)/2], sectors);

<h1 style="font-weight: bold; font-size:14px; line-height:1.5em;"><span style="color:#f56502; font-weight:bold">[3 points]</span> Task 3</h1>

Plot the 
<a style="color:#3c9591" href="http://en.wikipedia.org/wiki/Minimum_spanning_tree">minimum spanning tree</a>
based on the distance matrix $\boldsymbol{d}$ computed in **Task 1**, using different colors for nodes belonging to different sectors.

**Hint**: Use the
<a style="color:#3c9591" href="https://networkx.org/">networkx</a>
package as in script
<a style="color:#3c9591" href="../Python/1. Textbook support code/Machine Learning/31d. Clustering for the stock market/s_shrink_corr_clusters.ipynb">s_shrink_corr_clusters</a>.

In [ ]:
# minimum spanning tree
mst =  

##################################### plot #####################################
# colors by sectors
pos = kamada_kawai_layout(mst)
draw_networkx(mst, pos=pos, node_shape='.', node_size=1, with_labels=False)
idx = [i_s[int(l_s[k - 1] if k > 0 else 0):int(l_s[k])] for k in range(k_bar)]
[scatterplot(x=[pos[id][0] for id in idx[k]], y=[pos[id][1] for id in idx[k]])\
 for k in range(k_bar)];